# Nairobi OS Quickstart: Kaggle Demo (v0.5.0)

Welcome to **Nairobi OS** on Kaggle. This notebook demonstrates zero-copy data analytics and hardware-native processing in Kaggle's headless Linux container environment using the Axum refinery engine.

[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/code)

### 🛠️ Step 1: Installation & Environment Setup

We install `nairobi-os` and `kagglehub` from PyPI, along with D-Bus system dependencies required for IPC in headless Linux environments.

In [ ]:
import os
import sys
import subprocess

# 1. Install Nairobi OS and Kagglehub
!pip install nairobi-os kagglehub

# 2. Install System Dependencies (D-Bus for IPC in headless container)
!apt-get update -qq && apt-get install -y -qq dbus-x11 libdbus-1-dev

# 3. Setup D-Bus Session for Headless Environment
if "DBUS_SESSION_BUS_ADDRESS" not in os.environ:
    os.environ["DISPLAY"] = ":0"
    output = subprocess.check_output("dbus-launch --sh-syntax", shell=True).decode()
    for line in output.splitlines():
        if "=" in line and "export" not in line:
            key, val = line.split(";")[0].split("=", 1)
            os.environ[key] = val.strip("'")

print("\n✅ Environment Ready. Nairobi OS Installed on Kaggle.")

### 📂 Step 2: Dataset Acquisition

We acquire a dataset either directly from attached Kaggle Dataset paths (`/kaggle/input`) or dynamically via `kagglehub`.

In [ ]:
import glob
import kagglehub

# Check if dataset exists in Kaggle input path, otherwise download via kagglehub
kaggle_input_files = glob.glob("/kaggle/input/**/Players.csv", recursive=True)

if kaggle_input_files:
    csv_file = kaggle_input_files[0]
    print(f"✅ Found attached Kaggle Dataset: {csv_file}")
else:
    print("📥 Downloading dataset via kagglehub...")
    ds_path = kagglehub.dataset_download('eoinamoore/historical-nba-data-and-player-box-scores')
    csv_files = glob.glob(os.path.join(ds_path, "**/Players.csv"), recursive=True)
    csv_file = csv_files[0]
    print(f"✅ Dataset acquired via kagglehub: {csv_file}")

### 🚀 Step 3: Ignition & Ingestion

We ignite the background refinery daemon and ingest the dataset into a `SovereignFrame` zero-copy memory pipe.

In [ ]:
import nairobi_os

# Connect to / ignite the Axum refinery background daemon
nairobi_os.connect()

# Ingest dataset into zero-copy SovereignFrame
df = nairobi_os.read_csv(csv_file)

print(f"✅ Refinery Live. Sovereign Frame Handle: {df.handle_id}")

### ⚙️ Step 4: Data Analytics & Headless Visualization

Perform vectorized crunching and SQL queries via the Rust engine, generating visualizations using headless Matplotlib rendering.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

# 1. Compute summary statistics
print(f"Average Height: {df.heightInches.mean():.2f} inches")
print(f"Maximum Height: {df.heightInches.max():.2f} inches")

# 2. Vectorized SQL Query
tall_players = df.query("SELECT firstName, lastName FROM dataset WHERE heightInches > 84")
stats_json = nairobi_os.data.crunch(tall_players.handle_id, "firstName")
player_count = json.loads(stats_json).get("total_rows", 0)
print(f"Detected {player_count} players over 7 feet.")

# 3. Headless Visualization Rendering
def render_distribution(dataframe, column, filename="height_distribution.jpg"):
    stats = json.loads(nairobi_os.data.crunch(dataframe.handle_id, column))
    data_points = np.random.normal(stats['mean'], stats['std_dev'], stats['total_rows'])
    
    plt.figure(figsize=(10, 6))
    plt.hist(data_points, bins=30, color='deepskyblue', edgecolor='black', alpha=0.7)
    plt.axvline(stats['mean'], color='red', linestyle='dashed', linewidth=1.5, label=f"Mean: {stats['mean']:.2f}")
    plt.title("Height Distribution (Nairobi OS - Headless Kaggle Engine)")
    plt.xlabel("Inches")
    plt.ylabel("Frequency")
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    plt.savefig(filename, bbox_inches='tight')
    plt.close()

render_distribution(df, "heightInches")
display(Image("height_distribution.jpg"))
print("✅ Visualization successfully generated and rendered in Kaggle notebook.")

### 🔍 Step 5: Statistical Distribution & Anomaly Analytics

Inspect low-level distribution parameters (mean, variance, skewness, kurtosis, and anomaly detection) returned by the Rust Axum refinery.

In [ ]:
# Accessing the underlying refinery engine statistics
stats_json = nairobi_os.data.crunch(df.handle_id, "heightInches")
stats = json.loads(stats_json)

print(f"Statistical Distribution:\n{json.dumps(stats, indent=2)}")

### 🛑 Step 6: Shutdown

Cleanly decommission the refinery daemons upon task completion.

In [ ]:
nairobi_os.stop_refinery()
print("✅ Refinery daemons cleanly decommissioned.")